In [ ]:
import imaplib
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
import os
import re
import sys

In [ ]:
load_dotenv()
IMAP_SERVER = "export.imap.mail.yahoo.com"
EMAIL_ACCOUNT = os.getenv("YAHOO_EMAIL")
APP_PASSWORD = os.getenv("YAHOO_APP_PASSWORD") # see README for instructions on how to create the app password
print("Credentials loaded successfully!")

In [ ]:
def scan_all_yahoo_folders_accurate():
  """Scans all Yahoo Mail folders via IMAP to compile structural storage metrics.

    Connects to the Yahoo IMAP server to retrieve a comprehensive list of folders and 
    analyzes their contents without downloading full email payloads. It leverages 
    IMAP metadata (RFC822.SIZE and BODYSTRUCTURE) to efficiently calculate the exact 
    storage footprint of plain text bodies versus media attachments.

    Returns:
        pd.DataFrame: A tabular summary of the mailbox state, sorted by total storage 
        size in descending order. Features the following columns:
            - folder_name (str): The IMAP directory name.
            - email_count (int): Total number of messages in the folder.
            - attachment_count (int): Total number of distinct file attachments.
            - emails_size_mb (float): Storage size of email bodies/headers in megabytes.
            - attachments_size_mb (float): Storage size of all attachments in megabytes.
            - total_folder_mb (float): Combined total storage size in megabytes.
    """

  print(f"Connecting to {IMAP_SERVER} to fetch folder list...")
  mail = imaplib.IMAP4_SSL(IMAP_SERVER, 993)
  mail.login(EMAIL_ACCOUNT, APP_PASSWORD)

  # Fetch folder list directly from Yahoo IMAP
  status, folders = mail.list()
  if status != "OK":
    print("Failed to retrieve folders from Yahoo.")
    return pd.DataFrame()

  folder_list = []
  for f in folders:
    f_str = f.decode("utf-8", errors="ignore")
    # Parse IMAP list response (e.g., (\HasNoChildren) "/" "Folder Name")
    match = re.match(r'\((.*?)\)\s+"(.*?)"\s+(.*)', f_str)
    if match:
      folder_name = match.group(3).strip('"')
      folder_list.append(folder_name)

  print(f"Found {len(folder_list)} folders. Scanning sizes and attachments...\n")

  summary_data = []

  # Outer progress bar for the overall folder processing
  for folder_name in tqdm(
      folder_list, 
      desc="Total Progress", 
      file=sys.stdout, 
      position=0
  ):
    try:
      status, _ = mail.select(f'"{folder_name}"', readonly=True)
      if status != "OK":
        continue
    except Exception:
      continue

    status, messages = mail.search(None, "ALL")
    if status != "OK" or not messages[0]:
      summary_data.append({
          "folder_name": folder_name,
          "email_count": 0,
          "attachment_count": 0,
          "emails_size_mb": 0.0,
          "attachments_size_mb": 0.0,
          "total_folder_mb": 0.0,
      })
      continue

    email_ids = messages[0].split()
    email_count = len(email_ids)
    
    total_folder_bytes = 0
    total_attach_bytes = 0
    total_attach_count = 0

    # Helper function to parse IMAP metadata strings safely
    def process_imap_response(resp_text):
      nonlocal total_folder_bytes, total_attach_bytes, total_attach_count
      
      # 1. Extract Total Email Size
      size_match = re.search(r'RFC822\.SIZE\s+(\d+)', resp_text, re.IGNORECASE)
      if size_match:
        total_folder_bytes += int(size_match.group(1))
        
      # 2. Extract Parts from BODYSTRUCTURE to find attachments
      part_pattern = re.compile(
          r'"([\w\-]+)"\s+"([\w\-]+)"\s+(\(.*?\)|NIL)\s+(?:"[^"]*"|NIL)\s+(?:"[^"]*"|NIL)\s+"([\w\-]+)"\s+(\d+)', 
          re.IGNORECASE
      )
      
      for p in part_pattern.finditer(resp_text):
        p_type = p.group(1).upper()
        p_params = p.group(3).upper()
        p_size = int(p.group(5))
        
        # If the part is explicitly named (has a filename) or is a typical media format, flag as attachment
        if "NAME" in p_params or p_type in ["APPLICATION", "IMAGE", "VIDEO", "AUDIO"]:
          total_attach_bytes += p_size
          total_attach_count += 1

    if email_count > 0:
      chunk_size = 500
      chunks = [
          email_ids[i : i + chunk_size]
          for i in range(0, email_count, chunk_size)
      ]

      # Inner progress bar for chunks, leaves cleanly after each folder is done
      for chunk in tqdm(
          chunks, 
          desc=f"Scanning {folder_name[:15]}...", 
          leave=False, 
          file=sys.stdout, 
          position=1
      ):
        # Join IDs with commas for bulk fetching
        id_list_str = b",".join(chunk)
        try:
          # Fetch sizes and structure WITHOUT downloading the actual email content
          res, data = mail.fetch(id_list_str, "(RFC822.SIZE BODYSTRUCTURE)")
          if res == "OK":
            for response_part in data:
              if isinstance(response_part, tuple):
                process_imap_response(response_part[0].decode("utf-8", errors="ignore"))
              elif isinstance(response_part, bytes):
                process_imap_response(response_part.decode("utf-8", errors="ignore"))
        except Exception:
          # Fallback: if a batch chunk errors out, loop through individual IDs safely
          for e_id in chunk:
            try:
              res, data = mail.fetch(e_id, "(RFC822.SIZE BODYSTRUCTURE)")
              if res == "OK":
                for response_part in data:
                  if isinstance(response_part, tuple):
                    process_imap_response(response_part[0].decode("utf-8", errors="ignore"))
                  elif isinstance(response_part, bytes):
                    process_imap_response(response_part.decode("utf-8", errors="ignore"))
            except Exception:
              pass

    # Calculate final MB conversions
    folder_mb = total_folder_bytes / (1024 * 1024)
    attach_mb = total_attach_bytes / (1024 * 1024)
    body_mb = max(0.0, folder_mb - attach_mb) # Ensure no negatives from MIME boundaries

    summary_data.append({
        "folder_name": folder_name,
        "email_count": email_count,
        "attachment_count": total_attach_count,
        "emails_size_mb": round(body_mb, 2),
        "attachments_size_mb": round(attach_mb, 2),
        "total_folder_mb": round(folder_mb, 2),
    })
    
    # Use tqdm.write() so the terminal output slides neatly above the progress bars
    tqdm.write(
        f"✅ {folder_name.ljust(20)} | {str(email_count).rjust(5)} emails | "
        f"{str(total_attach_count).rjust(5)} attachments | "
        f"Body: {round(body_mb, 2):>7.2f} MB | Attach: {round(attach_mb, 2):>7.2f} MB | Total: {round(folder_mb, 2):>7.2f} MB"
    )

  mail.logout()
  print("\n🚀 Accurate folder size & attachment scan complete!")

  df_summary = pd.DataFrame(summary_data)
  return df_summary.sort_values(by="total_folder_mb", ascending=False)

In [ ]:
# Run the scan and display the top heaviest folders
df_folder_eda = scan_all_yahoo_folders_accurate()
df_folder_eda.head(20)

In [ ]:
df_folder_eda.to_csv("folder_sizes.csv", index=False, encoding="utf-8", header=True)

In [ ]:
df_folder_eda['folder_name'].to_csv("yahoo_folders.csv", index=False, encoding="utf-8", header=True)